# Column Prepare — explore แล้วแก้ทีละเรื่อง (interleaved)

ทำ prepare ครบทั้งหมด (text-clean + value-format + property-decompose) เขียนออก `processed/prepared/*.csv` โดยตรง — `eda_prepare.ipynb` เหลือแค่ EDA อย่างเดียว อ่านต่อจากที่นี่

แต่ละปัญหาเรียง **explore ก่อน แล้ว fix ตามทันที** แบ่ง 3 ช่วงใหญ่:
- **ช่วง 1 — Clean dirty text** (A/B/C): unicode/whitespace, typo, spelling variant
- **ช่วง 2 — Clean value formats** (D–I): thickness, stability, purity, material family, unit ผิด
- **ช่วง 3 — Prepare properties**: แตกชื่อ property, แปลงสเกล FF

ปิดท้ายด้วยหัวข้อที่**ยัง explore อย่างเดียว ยังไม่มี fix จริง** (process_type canon, atmosphere, unit-audit ที่เหลือ, cross-table)

## Setup

In [1]:
import os, re, json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

ROOT = Path.cwd() if (Path.cwd() / "processed" / "normalize").exists() else Path.cwd().parent
os.chdir(ROOT)
SRC = ROOT / "processed" / "normalize"
AUDIT = ROOT / "processed" / "audit"

# dtype=str -> pandas ไม่แปลงตัวเลข (100 -> 100.0) เขียนกลับได้ตรงต้นฉบับ นี่คือตารางที่จะถูกแก้จริง
tables = {p.stem: pd.read_csv(p, dtype=str) for p in sorted(SRC.glob("*.csv"))}
tables_raw = {name: df.copy() for name, df in tables.items()}   # สำเนา "ก่อน clean" ไว้เทียบตอนจบ

diff_log = []   # ทุกเซลล์ที่ถูกแก้ (PART text-clean): step, table, column, row, device_id, before, after

def log_and_apply(name: str, col: str, new: pd.Series, step: str) -> int:
    """เทียบคอลัมน์เดิมกับค่าใหม่ บันทึกเซลล์ที่ต่างลง diff_log แล้วค่อย apply"""
    df = tables[name]
    old = df[col]
    mask = old.notna() & (old != new)
    for idx in df.index[mask]:
        diff_log.append({"step": step, "table": name, "column": col, "row": int(idx),
                         "device_id": df.at[idx, "device_id"] if "device_id" in df.columns else "",
                         "before": old.at[idx], "after": new.at[idx]})
    df[col] = new.astype(object)
    return int(mask.sum())

def step_diff(step: str) -> pd.DataFrame:
    d = pd.DataFrame([r for r in diff_log if r["step"] == step],
                     columns=["step", "table", "column", "row", "device_id", "before", "after"])
    return (d.groupby(["table", "column", "before", "after"])
              .size().rename("n_cells").reset_index())

print("โหลด", len(tables), "ตาราง:", {k: v.shape for k, v in tables.items()})

โหลด 7 ตาราง: {'devices': (44, 4), 'layers': (352, 5), 'materials': (543, 7), 'process': (702, 9), 'properties': (563, 6), 'stability': (62, 10), 'vendors': (644, 4)}


### Master column inventory (ภาพรวมก่อนเริ่ม)

ตารางเดียวเห็นทุกคอลัมน์: ชนิด, %missing, จำนวนค่าไม่ซ้ำ, จำนวนกลุ่มที่สะกดต่าง (text), และอัตราที่ parse เป็นตัวเลขได้ (quant)

In [2]:
QUANT_COLS = {"thickness_nm","amount","parameter","value","error","sample_no",
              "temperature_c","humidity_pct","time_hr","retention_pct","t80_hr",
              "initial_pce","final_pce"}
SKIP_COLS = {"device_id","source_file","source_sheet","layer_id","step_order","doi_raw"}

def kind(col):
    return "quant" if col in QUANT_COLS else "text"

def tkey(v):
    s = str(v).strip().lower()
    s = re.sub(r"[\s\-_/():.,]+", "", s)
    s = s.replace("−", "-")
    return s

def variant_groups(series, keyfn=tkey):
    s = series.dropna().astype(str).map(str.strip)
    s = s[s != ""]
    groups = {}
    for v in s:
        groups.setdefault(keyfn(v), []).append(v)
    rows = []
    for k, vals in groups.items():
        c = Counter(vals)
        if len(c) > 1:
            rows.append({"canonical_key": k, "n_variants": len(c),
                         "variants": "  |  ".join(f"{x} ×{c[x]}" for x in sorted(c)),
                         "total_rows": sum(c.values())})
    out = pd.DataFrame(rows)
    return out.sort_values(["n_variants","total_rows"], ascending=False).reset_index(drop=True) if len(out) else out

def numeric_parse_rate(series):
    x = series.dropna().astype(str).str.strip()
    if len(x) == 0: return np.nan
    pure = x.str.fullmatch(r"-?\d+\.?\d*")
    return round(pure.mean(), 3)

rows = []
for name, df in tables.items():
    for col in df.columns:
        if col in SKIP_COLS: continue
        s = df[col]
        info = {"table": name, "column": col, "kind": kind(col),
                "n_nonnull": int(s.notna().sum()), "pct_missing": round(s.isna().mean()*100, 1),
                "n_unique": int(s.dropna().astype(str).str.strip().nunique())}
        if kind(col) == "text":
            vg = variant_groups(s)
            info["variant_groups"] = 0 if len(vg)==0 else int(len(vg))
            info["numeric_parse_rate"] = np.nan
        else:
            info["variant_groups"] = np.nan
            info["numeric_parse_rate"] = numeric_parse_rate(s)
        rows.append(info)
inventory = pd.DataFrame(rows)
inventory

,table,column,kind,n_nonnull,pct_missing,n_unique,variant_groups,numeric_parse_rate
0,devices,sample_no,quant,44,0.0,5,NaN,1.000
1,layers,layer_type,text,349,0.9,12,2.0,NaN
2,layers,material,text,315,10.5,65,0.0,NaN
3,layers,thickness_nm,quant,105,70.2,25,NaN,0.886
4,materials,material,text,511,5.9,61,0.0,NaN
5,materials,type,text,375,30.9,22,1.0,NaN
6,materials,solvent,text,337,37.9,37,6.0,NaN
7,materials,amount,quant,323,40.5,73,NaN,1.000
8,materials,unit,text,323,40.5,22,2.0,NaN
9,process,process_type,text,674,4.0,32,5.0,NaN


# ช่วง 1 — Clean dirty text

## A. Unicode ปน / whitespace เกิน — explore

In [3]:
bad_chars = sum(int(tables[n][c].fillna("").str.contains((chr(0x03BC) + chr(0x2212) + chr(0x00A0)).join(["[", "]"])).sum())
                for n in tables for c in tables[n].columns)
print(f"เจอ {bad_chars} เซลล์ ที่มี Greek mu (μ, U+03BC) / unicode minus (−, U+2212) / NBSP (U+00A0) ปนอยู่")
print("ตัวอย่างที่พบ (ถ้ามี):")
for n in tables:
    for c in tables[n].columns:
        s = tables[n][c].fillna("")
        hit = s[s.str.contains((chr(0x03BC) + chr(0x2212) + chr(0x00A0)).join(["[", "]"]))]
        if len(hit):
            print(f"  {n}.{c}: {list(hit.unique()[:5])}")

เจอ 25 เซลล์ ที่มี Greek mu (μ, U+03BC) / unicode minus (−, U+2212) / NBSP (U+00A0) ปนอยู่
ตัวอย่างที่พบ (ถ้ามี):
  materials.unit: ['μL']
  process.unit: ['μL, sec', 'μL']
  properties.unit: ['mA cm−2', 'μm']


## A. Unicode ปน / whitespace เกิน — fix

In [4]:
UNICODE_FIX = str.maketrans({chr(0x03BC): chr(0x00B5),   # Greek mu -> micro sign
                             chr(0x2212): "-",              # Unicode minus -> hyphen
                             chr(0x00A0): " "})             # NBSP -> space

def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", s.translate(UNICODE_FIX)).strip()

n_fixed = 0
for name, df in tables.items():
    for col in df.columns:
        after = df[col].map(lambda v: norm_text(v) if isinstance(v, str) else v)
        n_fixed += log_and_apply(name, col, after, "1-unicode/ws")
print("1-unicode/ws: แก้", n_fixed, "เซลล์")
print("before -> after ไม่ซ้ำ:")
step_diff("1-unicode/ws")

1-unicode/ws: แก้ 25 เซลล์
before -> after ไม่ซ้ำ:


,table,column,before,after,n_cells
0,materials,unit,μL,µL,2
1,process,unit,μL,µL,5
2,process,unit,"μL, sec","µL, sec",13
3,properties,unit,mA cm−2,mA cm-2,4
4,properties,unit,μm,µm,1


## B. Typo ที่รู้ตัวแล้ว — explore

In [5]:
TYPO_FIX = {
    ("properties", "property"): {
        "Trap densitiy": "Trap density",  # สะกดผิด
        "AverageVoc": "Average Voc",      # space หาย
    },
}
for (name, col), fixes in TYPO_FIX.items():
    for typo, correct in fixes.items():
        n = int((tables[name][col] == typo).sum())
        print(f"{name}.{col}: {typo!r} ผิด {n} แถว (ควรเป็น {correct!r})")

properties.property: 'Trap densitiy' ผิด 3 แถว (ควรเป็น 'Trap density')
properties.property: 'AverageVoc' ผิด 1 แถว (ควรเป็น 'Average Voc')


## B. Typo — fix

In [6]:
n_fixed = 0
for (name, col), fixes in TYPO_FIX.items():
    n_fixed += log_and_apply(name, col, tables[name][col].replace(fixes), "2-typo")
print("2-typo: แก้", n_fixed, "เซลล์")
print("before -> after ไม่ซ้ำ:")
step_diff("2-typo")

2-typo: แก้ 4 เซลล์
before -> after ไม่ซ้ำ:


,table,column,before,after,n_cells
0,properties,property,AverageVoc,Average Voc,1
1,properties,property,Trap densitiy,Trap density,3


## C. สะกดต่างกัน (case/วรรคตอน) — explore

In [7]:
TEXT_COLS = {
    "layers":     ["layer_type", "material"],
    "materials":  ["material", "type", "solvent", "unit"],
    "process":    ["process_type", "description", "unit"],
    "properties": ["property", "unit"],
    "stability":  ["environment", "encapsulation"],
    "vendors":    ["material", "vendor"],
}

PREFERRED = {
    ("process", "process_type", "spincoating"): "Spin coating",
    ("process", "process_type", "thermalevaporation"): "Thermal evaporation",
    ("materials", "solvent", "dmfdmso41vv"): "DMF/DMSO (4:1 v/v)",
    ("materials", "solvent", "dmfdmso3081vv"): "DMF/DMSO (3.08/1 v/v)",
    ("materials", "solvent", "dmfdmso41"): "DMF/DMSO (4:1)",
    ("properties", "property", "grainsize"): "Grain size",  # เสมอกัน 10:10
}

def variant_key(s: str) -> str:
    return re.sub(r"[\W_]", "", s.lower())

map_rows = []
for name, cols in TEXT_COLS.items():
    for col in cols:
        counts = Counter(tables[name][col].dropna())
        groups = {}
        for val, n in counts.items():
            k = variant_key(val)
            if re.search(r"[^\W\d_]", k):
                groups.setdefault(k, []).append((val, n))
        for k, variants in groups.items():
            if len(variants) < 2 and (name, col, k) not in PREFERRED:
                continue
            canonical = PREFERRED.get((name, col, k)) or \
                        max(variants, key=lambda v: (v[1], v[0]))[0]
            for val, n in variants:
                if val != canonical:
                    map_rows.append({"table": name, "column": col,
                                     "raw": val, "canonical": canonical, "n_rows": n,
                                     "rule": "override" if (name, col, k) in PREFERRED
                                             else "majority"})

var_map = (pd.DataFrame(map_rows,
                        columns=["table", "column", "raw", "canonical", "n_rows", "rule"])
             .sort_values(["table", "column", "canonical", "raw"])
             .reset_index(drop=True))
print(f"พบ {len(var_map)} แถว mapping ({var_map[['table','column','canonical']].drop_duplicates().shape[0]} กลุ่ม)")
var_map

พบ 44 แถว mapping (36 กลุ่ม)


,table,column,raw,canonical,n_rows,rule
0,layers,layer_type,HTL Treatment,HTL treatment,1,majority
1,layers,layer_type,PSK Treatment,PSK treatment,1,majority
2,materials,solvent,DMF /DMSO (3.08/1 V/V),DMF/DMSO (3.08/1 v/v),1,override
3,materials,solvent,DMF/DMSO (3.08/1 V/V),DMF/DMSO (3.08/1 v/v),1,override
4,materials,solvent,DMFDMSO (3.08/1 v/v),DMF/DMSO (3.08/1 v/v),2,override
5,materials,solvent,DMF/DMSO (4/1 V/V),DMF/DMSO (4:1 v/v),51,override
6,materials,solvent,DMF/DMSO (4:1 V/V),DMF/DMSO (4:1 v/v),43,override
7,materials,solvent,DMF:DMSO (4:1 v/v),DMF/DMSO (4:1 v/v),12,override
8,materials,solvent,DMF:DMSO (4:1),DMF/DMSO (4:1),10,override
9,materials,solvent,De-ionized water,Deionized water,3,majority


## C. สะกดต่างกัน — fix

In [8]:
report = []
for (name, col), g in var_map.groupby(["table", "column"]):
    mapping = dict(zip(g["raw"], g["canonical"]))
    before_unique = tables[name][col].nunique()
    n = log_and_apply(name, col, tables[name][col].replace(mapping), "3-variant")
    after_unique = tables[name][col].nunique()
    report.append({"table": name, "column": col, "n_cells": n,
                   "n_unique_before": before_unique, "n_unique_after": after_unique})
print("สรุปต่อคอลัมน์:")
display(pd.DataFrame(report))
print("before -> after ไม่ซ้ำ (ทุกคู่ raw -> canonical):")
step_diff("3-variant")

สรุปต่อคอลัมน์:


,table,column,n_cells,n_unique_before,n_unique_after
0,layers,layer_type,2,12,10
1,materials,solvent,126,37,27
2,materials,type,2,22,21
3,materials,unit,14,21,18
4,process,description,3,47,44
5,process,process_type,293,32,26
6,process,unit,15,33,28
7,properties,property,30,97,91
8,properties,unit,17,26,23
9,stability,environment,2,23,22


before -> after ไม่ซ้ำ (ทุกคู่ raw -> canonical):


,table,column,before,after,n_cells
0,layers,layer_type,HTL Treatment,HTL treatment,1
1,layers,layer_type,PSK Treatment,PSK treatment,1
2,materials,solvent,DMF /DMSO (3.08/1 V/V),DMF/DMSO (3.08/1 v/v),1
3,materials,solvent,DMF/DMSO (3.08/1 V/V),DMF/DMSO (3.08/1 v/v),1
4,materials,solvent,DMF/DMSO (4/1 V/V),DMF/DMSO (4:1 v/v),51
5,materials,solvent,DMF/DMSO (4:1 V/V),DMF/DMSO (4:1 v/v),43
6,materials,solvent,DMF:DMSO (4:1 v/v),DMF/DMSO (4:1 v/v),12
7,materials,solvent,DMF:DMSO (4:1),DMF/DMSO (4:1),10
8,materials,solvent,DMFDMSO (3.08/1 v/v),DMF/DMSO (3.08/1 v/v),2
9,materials,solvent,De-ionized water,Deionized water,3


## สรุป + save audit + verify (จบช่วง 1)

In [9]:
diff_cells = pd.DataFrame(diff_log,
                          columns=["step", "table", "column", "row", "device_id",
                                   "before", "after"])
recount = sum(int((tables_raw[n][c].notna() & (tables_raw[n][c] != tables[n][c])).sum())
              for n in tables for c in tables[n].columns)
n_cells_logged = len(diff_cells[["table", "column", "row"]].drop_duplicates())
ok = "ตรงกัน ✓" if recount == n_cells_logged else "ไม่ตรง — ต้องตรวจ!"
print(f"เซลล์ที่แก้ (นับจาก log)      : {n_cells_logged}")
print(f"เซลล์ที่แก้ (นับอิสระจาก raw) : {recount}  ({ok})")

เซลล์ที่แก้ (นับจาก log)      : 566
เซลล์ที่แก้ (นับอิสระจาก raw) : 566  (ตรงกัน ✓)


In [10]:
AUDIT.mkdir(parents=True, exist_ok=True)
var_map.to_csv(AUDIT / "text_cleanup_map.csv", index=False)
diff_cells.to_csv(AUDIT / "clean_diff_cells.csv", index=False)
print("wrote", AUDIT / "text_cleanup_map.csv")
print("wrote", AUDIT / "clean_diff_cells.csv")

wrote D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\audit\text_cleanup_map.csv
wrote D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\audit\clean_diff_cells.csv


In [11]:
leftover = []
for name, cols in TEXT_COLS.items():
    for col in cols:
        groups = {}
        for val in tables[name][col].dropna():
            k = variant_key(val)
            if re.search(r"[^\W\d_]", k):
                groups.setdefault(k, set()).add(val)
        leftover += [(name, col, sorted(v)) for v in groups.values() if len(v) > 1]

bad_chars = sum(int(tables[n][c].fillna("").str.contains((chr(0x03BC) + chr(0x2212) + chr(0x00A0)).join(["[", "]"])).sum())
                for n in tables for c in tables[n].columns)
same_shape = all(tables[n].shape == tables_raw[n].shape for n in tables)

print("1) ไม่มี variant กลุ่มไหนหลงเหลือ :", not leftover)
print("2) ไม่มีอักขระ mu/minus/NBSP หลงเหลือ :", bad_chars == 0)
print("3) shape ตารางไม่เปลี่ยน :", same_shape)
assert not leftover and bad_chars == 0 and same_shape, "VERIFY FAILED"
print("ช่วง 1 (text-clean) verify ผ่านหมด ✓")

1) ไม่มี variant กลุ่มไหนหลงเหลือ : True
2) ไม่มีอักขระ mu/minus/NBSP หลงเหลือ : True
3) shape ตารางไม่เปลี่ยน : True
ช่วง 1 (text-clean) verify ผ่านหมด ✓


# ช่วง 2 — Clean value formats

ต่อจากช่วง 1 — ใช้ตาราง `tables` ใน memory ต่อเลย (ไม่มีโฟลเดอร์กลาง). กติกาต่างจากช่วง 1: ไม่ทับคอลัมน์เดิม — เพิ่มคอลัมน์ใหม่ `_parsed`/`_note` แทนเสมอ

## D. ภาพรวมคอลัมน์ตัวเลขทั้งหมด — explore

In [12]:
tables_after_text = {name: df.copy() for name, df in tables.items()}   # snapshot ไว้ verify ท้ายช่วง 2

CONFIG = {
    "RT_CELSIUS": 25.0,
    "ANGSTROM_PER_NM": 10.0,
    "RANGE_REPRESENTATIVE": "mid",
}
NUM = r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?"

def is_num(s) -> bool:
    return isinstance(s, str) and re.fullmatch(NUM, s.strip()) is not None

audit_rows = []  # ทุกแถวที่ถูกตีความจะถูกบันทึกลงนี่

NUM_COLS = {
    "layers":     ["thickness_nm"],
    "materials":  ["amount"],
    "properties": ["value", "error"],
    "stability":  ["temperature_c", "humidity_pct", "time_hr", "retention_pct",
                   "t80_hr", "initial_pce", "final_pce"],
    "vendors":    ["purity"],
}
scan = []
for name, cols in NUM_COLS.items():
    for col in cols:
        s = tables[name][col].dropna().str.strip()
        s = s[s != ""]
        bad = s[~s.map(is_num)]
        scan.append({"table": name, "column": col,
                     "n_filled": len(s), "n_bad": len(bad),
                     "bad_examples": ", ".join(sorted(bad.unique())[:5])})
scan = pd.DataFrame(scan).sort_values("n_bad", ascending=False).reset_index(drop=True)
print(f"แถวที่ parse ไม่ได้รวม: {scan.n_bad.sum()}")
scan

แถวที่ parse ไม่ได้รวม: 79


,table,column,n_filled,n_bad,bad_examples
0,vendors,purity,346,48,"20 wt, 9 - 15 Ω □-1, >97, >99.0, >99.99"
1,layers,thickness_nm,105,12,"1200 Å, 4, 100, 400 Å, 400-450, 90 Å"
2,stability,humidity_pct,26,12,"25 ± 5, 30-50, 45–55, 50 - 60, 50 ± 5"
3,stability,temperature_c,48,6,RT
4,stability,retention_pct,50,1,< 55
5,properties,error,74,0,
6,materials,amount,323,0,
7,properties,value,544,0,
8,stability,time_hr,59,0,
9,stability,t80_hr,0,0,


## E. `layers.thickness_nm` — explore

In [13]:
s = tables["layers"]["thickness_nm"].dropna().str.strip()
bad = s[(s != "") & ~s.map(is_num)]
print(f"{len(bad)} แถว parse ตรงๆ ไม่ได้ (หน่วย Å ปน / ช่วงค่า / หลายชั้นย่อยในเซลล์เดียว):")
tables["layers"].loc[bad.index, ["device_id", "layer_type", "material", "thickness_nm"]]

12 แถว parse ตรงๆ ไม่ได้ (หน่วย Å ปน / ช่วงค่า / หลายชั้นย่อยในเซลล์เดียว):


,device_id,layer_type,material,thickness_nm
3,10.1002admi.202500231_Sample1,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
11,10.1002admi.202500231_Sample2,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
19,10.1002admi.202500231_Sample3,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
27,10.1002admi.202500231_Sample4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
119,10_1002_solr_202200422_Sample1,Cathode,Cr + Au,"4, 100"
127,10_1002_solr_202200422_Sample2,Cathode,Cr + Au,"4, 100"
134,10_1002_solr_202400534_Sample1,ETL treatment (1),PCBM,400 Å
135,10_1002_solr_202400534_Sample1,ETL treatment (2),BCP,90 Å
136,10_1002_solr_202400534_Sample1,Cathode,Cu,1200 Å
143,10_1002_solr_202400534_Sample2,ETL treatment (1),PCBM,400 Å


## E. `layers.thickness_nm` — fix

In [14]:
def parse_thickness(v):
    """คืน (parsed, vmin, vmax, note) — note ว่าง = ตัวเลขปกติไม่ต้องตีความ"""
    if not isinstance(v, str) or not v.strip():
        return (None, None, None, "")
    s = v.strip()
    if is_num(s):                                     # ตัวเลขปกติ
        x = float(s)
        return (x, x, x, "")
    m = re.fullmatch(rf"({NUM})\s*Å", s)              # หน่วย Å
    if m:
        x = float(m[1]) / CONFIG["ANGSTROM_PER_NM"]
        return (x, x, x, f"แปลงหน่วย: {s} = {x:g} nm (1 Å = 0.1 nm)")
    m = re.fullmatch(rf"({NUM})\s*[-–]\s*({NUM})", s)  # ช่วง a-b
    if m:
        a, b = float(m[1]), float(m[2])
        return ((a + b) / 2, a, b, f"ช่วงค่า: ใช้ค่ากลาง {(a + b) / 2:g} (min={a:g}, max={b:g})")
    m = re.fullmatch(rf"({NUM})\s*,\s*({NUM})", s)     # หลายชั้นย่อย a, b
    if m:
        a, b = float(m[1]), float(m[2])
        return (a + b, min(a, b), max(a, b),
                f"หลายชั้นย่อยในเซลล์เดียว: ผลรวม {a + b:g} nm (ชั้นย่อย {a:g} + {b:g})")
    return (None, None, None, f"parse ไม่ได้: {s!r}")

lay = tables["layers"]
parsed = lay["thickness_nm"].map(parse_thickness)
lay["thickness_nm_parsed"] = parsed.map(lambda t: t[0])
lay["thickness_nm_min"] = parsed.map(lambda t: t[1])
lay["thickness_nm_max"] = parsed.map(lambda t: t[2])
lay["thickness_nm_note"] = parsed.map(lambda t: t[3])

hit = lay["thickness_nm_note"] != ""
for _, r in lay[hit].iterrows():
    audit_rows.append({"table": "layers", "column": "thickness_nm",
                       "device_id": r["device_id"], "raw": r["thickness_nm"],
                       "parsed": r["thickness_nm_parsed"], "vmin": r["thickness_nm_min"],
                       "vmax": r["thickness_nm_max"], "extra": r["layer_type"],
                       "note": r["thickness_nm_note"]})

print(f"แถวที่ถูกตีความ: {int(hit.sum())}")
lay.loc[hit, ["device_id", "layer_type", "material", "thickness_nm",
              "thickness_nm_parsed", "thickness_nm_min", "thickness_nm_max",
              "thickness_nm_note"]].rename(
    columns={"thickness_nm": "before (raw)", "thickness_nm_parsed": "after (parsed)"})

แถวที่ถูกตีความ: 12


,device_id,layer_type,material,before (raw),after (parsed),thickness_nm_min,thickness_nm_max,thickness_nm_note
3,10.1002admi.202500231_Sample1,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
11,10.1002admi.202500231_Sample2,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
19,10.1002admi.202500231_Sample3,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
27,10.1002admi.202500231_Sample4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
119,10_1002_solr_202200422_Sample1,Cathode,Cr + Au,"4, 100",104.0,4.0,100.0,หลายชั้นย่อยในเซลล์เดียว: ผลรวม 104 nm (ชั้นย่อย 4 + 100)
127,10_1002_solr_202200422_Sample2,Cathode,Cr + Au,"4, 100",104.0,4.0,100.0,หลายชั้นย่อยในเซลล์เดียว: ผลรวม 104 nm (ชั้นย่อย 4 + 100)
134,10_1002_solr_202400534_Sample1,ETL treatment (1),PCBM,400 Å,40.0,40.0,40.0,แปลงหน่วย: 400 Å = 40 nm (1 Å = 0.1 nm)
135,10_1002_solr_202400534_Sample1,ETL treatment (2),BCP,90 Å,9.0,9.0,9.0,แปลงหน่วย: 90 Å = 9 nm (1 Å = 0.1 nm)
136,10_1002_solr_202400534_Sample1,Cathode,Cu,1200 Å,120.0,120.0,120.0,แปลงหน่วย: 1200 Å = 120 nm (1 Å = 0.1 nm)
143,10_1002_solr_202400534_Sample2,ETL treatment (1),PCBM,400 Å,40.0,40.0,40.0,แปลงหน่วย: 400 Å = 40 nm (1 Å = 0.1 nm)


## F. `stability` (temperature/humidity/retention) — explore

In [15]:
stb0 = tables["stability"]
for col in ["temperature_c", "humidity_pct", "retention_pct"]:
    s = stb0[col].dropna().str.strip()
    bad = s[(s != "") & ~s.map(is_num)]
    print(f"{col}: {len(bad)} แถว parse ตรงๆ ไม่ได้ -> {sorted(bad.unique())}")

temperature_c: 6 แถว parse ตรงๆ ไม่ได้ -> ['RT']
humidity_pct: 12 แถว parse ตรงๆ ไม่ได้ -> ['25 ± 5', '30-50', '45–55', '50 - 60', '50 ± 5', '50–60', '60 ± 5']
retention_pct: 1 แถว parse ตรงๆ ไม่ได้ -> ['< 55']


## F. `stability` — fix

In [16]:
def parse_temperature(v):
    if not isinstance(v, str) or not v.strip():
        return (None, "")
    s = v.strip()
    if is_num(s):
        return (float(s), "")
    if s.upper() == "RT":
        return (CONFIG["RT_CELSIUS"],
                f"RT (room temperature) -> {CONFIG['RT_CELSIUS']:g} C [assumption: CONFIG]")
    return (None, f"parse ไม่ได้: {s!r}")

def parse_range_pm(v):
    """สำหรับ humidity: ตัวเลข | ช่วง a-b | c ± t -> (parsed, min, max, note)"""
    if not isinstance(v, str) or not v.strip():
        return (None, None, None, "")
    s = v.strip()
    if is_num(s):
        x = float(s)
        return (x, x, x, "")
    m = re.fullmatch(rf"({NUM})\s*[-–]\s*({NUM})", s)
    if m:
        a, b = float(m[1]), float(m[2])
        return ((a + b) / 2, a, b, f"ช่วงค่า: ใช้ค่ากลาง {(a + b) / 2:g} (min={a:g}, max={b:g})")
    m = re.fullmatch(rf"({NUM})\s*±\s*({NUM})", s)
    if m:
        c, t = float(m[1]), float(m[2])
        return (c, c - t, c + t, f"ค่ากลาง±คลาดเคลื่อน: {c:g} (min={c - t:g}, max={c + t:g})")
    return (None, None, None, f"parse ไม่ได้: {s!r}")

def parse_censored(v):
    """สำหรับ retention: ตัวเลข | '< x' -> (parsed, censor, note)"""
    if not isinstance(v, str) or not v.strip():
        return (None, "", "")
    s = v.strip()
    if is_num(s):
        return (float(s), "", "")
    m = re.fullmatch(rf"([<>])\s*({NUM})", s)
    if m:
        return (float(m[2]), m[1],
                f"ค่า censored: รู้แค่ว่า {'ต่ำ' if m[1] == '<' else 'สูง'}กว่า {m[2]} — ห้ามใช้เป็นค่าจริงโดยไม่ดูธง censor")
    return (None, "", f"parse ไม่ได้: {s!r}")

stb = tables["stability"]

t_ = stb["temperature_c"].map(parse_temperature)
stb["temperature_c_parsed"] = t_.map(lambda x: x[0])
stb["temperature_c_note"] = t_.map(lambda x: x[1])

h = stb["humidity_pct"].map(parse_range_pm)
stb["humidity_pct_parsed"] = h.map(lambda x: x[0])
stb["humidity_pct_min"] = h.map(lambda x: x[1])
stb["humidity_pct_max"] = h.map(lambda x: x[2])
stb["humidity_pct_note"] = h.map(lambda x: x[3])

r = stb["retention_pct"].map(parse_censored)
stb["retention_pct_parsed"] = r.map(lambda x: x[0])
stb["retention_pct_censor"] = r.map(lambda x: x[1])
stb["retention_pct_note"] = r.map(lambda x: x[2])

for col, pcol, note_col in [("temperature_c", "temperature_c_parsed", "temperature_c_note"),
                            ("humidity_pct", "humidity_pct_parsed", "humidity_pct_note"),
                            ("retention_pct", "retention_pct_parsed", "retention_pct_note")]:
    hit = stb[note_col] != ""
    for _, row in stb[hit].iterrows():
        audit_rows.append({"table": "stability", "column": col,
                           "device_id": row["device_id"], "raw": row[col],
                           "parsed": row[pcol],
                           "vmin": row.get("humidity_pct_min") if col == "humidity_pct" else None,
                           "vmax": row.get("humidity_pct_max") if col == "humidity_pct" else None,
                           "extra": row.get("retention_pct_censor", "") if col == "retention_pct" else "",
                           "note": row[note_col]})

hit_any = (stb["temperature_c_note"] != "") | (stb["humidity_pct_note"] != "") | (stb["retention_pct_note"] != "")
print(f"แถวที่ถูกตีความ: {int(hit_any.sum())}")
stb.loc[hit_any, ["device_id", "environment", "temperature_c", "temperature_c_parsed",
                  "humidity_pct", "humidity_pct_parsed", "humidity_pct_min", "humidity_pct_max",
                  "retention_pct", "retention_pct_parsed", "retention_pct_censor"]]

แถวที่ถูกตีความ: 18


,device_id,environment,temperature_c,temperature_c_parsed,humidity_pct,humidity_pct_parsed,humidity_pct_min,humidity_pct_max,retention_pct,retention_pct_parsed,retention_pct_censor
8,10.1016j.nanoen.2021.106285_Sample1,Thermal Aging,85,85.0,50 - 60,55.0,50.0,60.0,NaN,NaN,
9,10.1016j.nanoen.2021.106285_Sample2,Thermal Aging,85,85.0,50–60,55.0,50.0,60.0,NaN,NaN,
11,10.1016j.nanoen.2021.106285_Sample3,Ambient MPP,NaN,NaN,45–55,50.0,45.0,55.0,90.0,90.00,
12,10.1016j.nanoen.2021.106285_Sample3,Thermal Aging,85.0,85.0,50–60,55.0,50.0,60.0,NaN,NaN,
14,10.1021_acsami.5c16972_Sample1,ambient,25,25.0,30-50,40.0,30.0,50.0,38,38.00,
16,10.1021_acsami.5c16972_Sample2,ambient,25,25.0,30-50,40.0,30.0,50.0,70,70.00,
18,10.1021_acsami.5c16972_Sample3,ambient,25,25.0,30-50,40.0,30.0,50.0,85,85.00,
20,10_1002_solr_202200422_Sample1,Ambient air,25,25.0,60 ± 5,60.0,55.0,65.0,62,62.00,
35,10_1016_j_solener_2025_113286_Sample1,ambient,RT,25.0,35,35.0,35.0,35.0,< 55,55.00,<
36,10_1016_j_solener_2025_113286_Sample2,ambient,RT,25.0,35,35.0,35.0,35.0,80,80.00,


## G. `vendors.purity` — explore

In [17]:
s = tables["vendors"]["purity"].dropna().str.strip()
bad = s[(s != "") & ~s.map(is_num)]
print(f"{len(bad)} แถว ไม่ใช่ตัวเลขล้วน — ปนกันหลายชนิด (qualifier/เกรด/wt%/sheet-resistance/missing):")
bad.value_counts()

48 แถว ไม่ใช่ตัวเลขล้วน — ปนกันหลายชนิด (qualifier/เกรด/wt%/sheet-resistance/missing):


purity
Not provided    17
>97              8
Anhydrous        6
9 - 15 Ω □-1     4
>99.99           4
20 wt            3
>99.0            2
≥99.99           2
≥99.5            2
Name: count, dtype: int64

## G. `vendors.purity` — fix

In [18]:
def parse_purity(v):
    """คืน (pct, qualifier, kind, note)"""
    if not isinstance(v, str) or not v.strip() or v.strip() == "Not provided":
        return (None, "", "missing", "")
    s = v.strip()
    if is_num(s):
        return (float(s), "=", "percent", "")
    m = re.fullmatch(rf"([>≥])\s*({NUM})", s)
    if m:
        return (float(m[2]), m[1], "percent_lower_bound",
                f"qualifier {m[1]}: ความบริสุทธิ์อย่างน้อย {m[2]}%")
    if s.lower() == "anhydrous":
        return (None, "", "grade_descriptor",
                "เกรดตัวทำละลาย 'ปราศจากน้ำ' — ไม่ใช่ตัวเลขความบริสุทธิ์")
    m = re.fullmatch(rf"({NUM})\s*wt", s)
    if m:
        return (None, "", "concentration_wt_pct",
                f"ความเข้มข้น {m[1]} wt% ของ dispersion — ไม่ใช่ purity (ข้อมูลผิดชนิด)")
    if "Ω" in s or "Ω" in s or "□" in s:
        return (None, "", "sheet_resistance_misfiled",
                f"sheet resistance ของ substrate ({s}) ถูกกรอกในช่อง purity — Ω ในไฟล์คือ U+2126 OHM SIGN")
    return (None, "", "unrecognized", f"จำแนกไม่ได้: {s!r}")

ven = tables["vendors"]
pu = ven["purity"].map(parse_purity)
ven["purity_pct"] = pu.map(lambda x: x[0])
ven["purity_qualifier"] = pu.map(lambda x: x[1])
ven["purity_kind"] = pu.map(lambda x: x[2])
_purity_note = pu.map(lambda x: x[3])

hit = ven["purity"].notna() & (ven["purity"].str.strip() != "") & ~ven["purity"].fillna("").map(is_num)
for idx in ven.index[hit]:
    audit_rows.append({"table": "vendors", "column": "purity",
                       "device_id": ven.at[idx, "device_id"], "raw": ven.at[idx, "purity"],
                       "parsed": ven.at[idx, "purity_pct"], "vmin": None, "vmax": None,
                       "extra": ven.at[idx, "purity_kind"], "note": _purity_note[idx]})

print("จำแนกชนิดได้ดังนี้ (นับเฉพาะแถวที่มีค่า):")
display(ven.loc[ven["purity"].notna() & (ven["purity"].str.strip() != ""), "purity_kind"]
           .value_counts().rename("n_rows").to_frame())

print("แถวที่ไม่ใช่ percent ปกติ (ตัวอย่างการจำแนก):")
ven.loc[hit, ["device_id", "material", "purity", "purity_pct",
              "purity_qualifier", "purity_kind"]].drop_duplicates(
    subset=["material", "purity", "purity_kind"]).reset_index(drop=True).rename(
    columns={"purity": "before (raw)", "purity_pct": "after (pct)"})

จำแนกชนิดได้ดังนี้ (นับเฉพาะแถวที่มีค่า):


,n_rows
purity_kind,
percent,298
percent_lower_bound,18
missing,17
grade_descriptor,6
sheet_resistance_misfiled,4
concentration_wt_pct,3


แถวที่ไม่ใช่ percent ปกติ (ตัวอย่างการจำแนก):


,device_id,material,before (raw),after (pct),purity_qualifier,purity_kind
0,10.1002admi.202500231_Sample1,ITO coated glass,9 - 15 Ω □-1,NaN,,sheet_resistance_misfiled
1,10.1002admi.202500231_Sample1,MABr,>99.99,99.99,>,percent_lower_bound
2,10.1002admi.202500231_Sample1,NiNO3.6H2O,>97,97.00,>,percent_lower_bound
3,10.1002admi.202500231_Sample1,NaOH,>97,97.00,>,percent_lower_bound
4,10.1016j.nanoen.2021.106285_Sample1,Al2O3,20 wt,NaN,,concentration_wt_pct
5,10_1002_solr_202400534_Sample1,BCP,>99.0,99.00,>,percent_lower_bound
6,10_1002_solr_202400534_Sample1,CsI,≥99.99,99.99,≥,percent_lower_bound
7,10_1002_solr_202400534_Sample1,PEAI,≥99.5,99.50,≥,percent_lower_bound
8,10_1016_j_solener_2025_113286_Sample1,DMSO,Anhydrous,NaN,,grade_descriptor
9,10_1016_j_solener_2025_113286_Sample1,Chlorobenzene,Anhydrous,NaN,,grade_descriptor


## H. จับกลุ่ม material family — explore

ชื่อวัสดุปรากฏใน 3 ตาราง (`layers`, `materials`, `vendors`) เช่น `NiOx / NiOx NPs / NiOx Nanoparticles` = ตระกูลเดียวกัน (NiOx)

In [19]:
def material_family(x):
    s = str(x).lower()
    if "nio" in s: return "NiOx"
    if s.startswith("c60") or s=="c60": return "C60"
    if "bcp" in s: return "BCP"
    if "pbi2" in s: return "PbI2"
    if "pedot" in s: return "PEDOT:PSS"
    if "spiro" in s: return "Spiro-OMeTAD"
    if any(k in s for k in ["pacz","2pacz","me-2pacz","mecz"]): return "SAM (xPACz)"
    return None

mats_preview = pd.concat([tables["layers"][["material"]].assign(src="layers"),
                          tables["materials"][["material"]].assign(src="materials"),
                          tables["vendors"][["material"]].assign(src="vendors")], ignore_index=True)
mats_preview = mats_preview.dropna()
mats_preview["material"] = mats_preview["material"].astype(str).str.strip()
mats_preview["family"] = mats_preview["material"].map(material_family)
fam_preview = (mats_preview.dropna(subset=["family"]).groupby("family")["material"]
                  .agg(lambda s: "  |  ".join(f"{v} ×{c}" for v,c in Counter(s).most_common()))
                  .reset_index(name="spellings_found"))
fam_preview["n_spellings"] = fam_preview["spellings_found"].str.count(r"\|") + 1
print("ตระกูลวัสดุที่มีหลายชื่อ (ยังไม่ apply เป็นคอลัมน์):")
fam_preview.sort_values("n_spellings", ascending=False)

ตระกูลวัสดุที่มีหลายชื่อ (ยังไม่ apply เป็นคอลัมน์):


,family,spellings_found,n_spellings
2,NiOx,NiOx ×51 | NiOx NPs ×35 | NiO target ×6 | NiOx Nanoparticles ×5 | NiOx NPs ink ×2 | Cu:NiO ×1 | NiOx:TTT...,9
4,SAM (xPACz),Me-4PACz ×10 | 2PACz ×4 | Me-2PACz ×4 | Me-2PACz (SAM) ×2 | Me-4PACz + PFPTFB ×1 | 2PACz SAM ×1 | Me-4PA...,9
0,BCP,"BCP ×111 | PbI2, BCP ×5 | BCP (bathocuproine) ×2 | BCP + SnO2 ×1",4
1,C60,C60 ×52,1
3,PbI2,PbI2 ×78,1


## H. material family — fix

**หลักการ**: ต่างจาก C (variant-merge) ตรงที่ C รวมได้แค่ค่าที่**สะกดต่างกันแต่ความหมายเดียวกันเป๊ะ**
(เช่น `SpinCoating` = `Spin coating`) — แต่ `NiOx`, `NiOx NPs`, `NiOx Nanoparticles` เป็นคนละ string
กันจริง ๆ (ไม่ใช่แค่สะกดต่าง) C เลยไม่รวมให้ ต้องมาจับ**กลุ่มตระกูลเคมี** (chemistry family) แทน
ด้วยการเช็ค keyword ในชื่อ (เช่น มีคำว่า "nio" ⇒ ตระกูล NiOx) ครอบคลุม 7 ตระกูลที่เจอบ่อยในชุดข้อมูลนี้
(NiOx, C60, BCP, PbI2, PEDOT:PSS, Spiro-OMeTAD, SAM/xPACz)

**ไม่ทับข้อมูลเดิม** — เพิ่มคอลัมน์ใหม่ `material_family` ให้ `layers`/`materials`/`vendors` เท่านั้น
(หลักการเดียวกับ E/F/G/I) คอลัมน์ `material` เดิมยังอยู่ครบ

**ทำไมมี NaN เยอะ** — วัสดุส่วนใหญ่ในตาราง `materials`/`vendors` เป็นสารตั้งต้น/ตัวทำละลาย (เช่น PbBr2, DMF,
CsI, NaOH) ที่ไม่ได้อยู่ใน 7 ตระกูลนี้ — `material_family` ตั้งใจครอบคลุมเฉพาะวัสดุที่ทำหน้าที่เป็น
**ชั้นฟังก์ชัน** (HTL/ETL) ของเซลล์ที่มักถูกเขียนชื่อหลายแบบ ไม่ใช่ทุกวัสดุต้องมี family

**ประโยชน์**: วิเคราะห์/กรองข้อมูลระดับ "ตระกูล" ได้ (เช่น เทียบ NiOx ทุกแบบเป็นกลุ่มเดียว) โดยไม่ต้องเสียชื่อเฉพาะ
(`NiOx NPs` vs `NiOx Nanoparticles`) ที่อาจมีนัยสำคัญ (ขนาดอนุภาค, กระบวนการสังเคราะห์) ไปจากคอลัมน์ `material` เดิม


In [20]:
for name in ["layers", "materials", "vendors"]:
    tables[name]["material_family"] = tables[name]["material"].map(material_family)

mats = pd.concat([tables[n][["material", "material_family"]].assign(src=n)
                  for n in ["layers", "materials", "vendors"]], ignore_index=True)
classified = mats.dropna(subset=["material_family"])
fam = (classified.groupby("material_family")["material"]
                  .agg(lambda s: "  |  ".join(f"{v} ×{c}" for v, c in Counter(s).most_common()))
                  .reset_index(name="spellings_found"))
fam["n_rows"] = classified.groupby("material_family").size().values
print(f"จัดกลุ่มได้ {len(classified)} / {len(mats)} แถว (รวม 3 ตาราง) เข้า {fam['material_family'].nunique()} family")
display(fam.sort_values("n_rows", ascending=False))
print("before (material) -> after (material_family) รายค่า:")
classified.drop_duplicates(subset=["material", "material_family"]).sort_values(
    ["material_family", "material"]).rename(
    columns={"material": "before", "material_family": "after"})[["before", "after", "src"]]

จัดกลุ่มได้ 377 / 1539 แถว (รวม 3 ตาราง) เข้า 5 family


,material_family,spellings_found,n_rows
0,BCP,"BCP ×111 | PbI2, BCP ×5 | BCP (bathocuproine) ×2 | BCP + SnO2 ×1",119
2,NiOx,NiOx ×51 | NiOx NPs ×35 | NiO target ×6 | NiOx Nanoparticles ×5 | NiOx NPs ink ×2 | Cu:NiO ×1 | NiOx:TTT...,103
3,PbI2,PbI2 ×78,78
1,C60,C60 ×52,52
4,SAM (xPACz),Me-4PACz ×10 | 2PACz ×4 | Me-2PACz ×4 | Me-2PACz (SAM) ×2 | Me-4PACz + PFPTFB ×1 | 2PACz SAM ×1 | Me-4PA...,25


before (material) -> after (material_family) รายค่า:


,before,after,src
6,BCP,BCP,layers
1120,BCP (bathocuproine),BCP,vendors
70,BCP + SnO2,BCP,layers
1180,"PbI2, BCP",BCP,vendors
5,C60,C60,layers
105,Cu:NiO,NiOx,layers
515,Cu:NiO (NC10),NiOx,materials
503,NiO,NiOx,materials
693,NiO target,NiOx,materials
33,NiOx,NiOx,layers


## I. `properties` unit ผิด (Voc/Jsc ติด `%`, mobility หน่วย m²) — explore

พอร์ตมาจาก §5.3 เดิมของ `column_audit.ipynb` (ตอนนี้อยู่ในหัวข้อ "ยัง explore อย่างเดียว" ด้านล่างสุดของสมุดนี้)
เจอ 2 ปัญหาคนละแบบ ปนกันอยู่ในคอลัมน์ `properties.unit`:

### ปัญหาที่ 1 — Voc/Jsc ติด unit `'%'` (12 แถว)

ทั้ง 12 แถวมาจาก **paper เดียว** (`10.1016j.jechem.2021.11.029`, ทั้ง 2 samples) — คอลัมน์ `unit` เขียนว่า `%`
แต่ค่า `value` (Voc ≈ 1.06–1.12, Jsc ≈ 20.7–22.1) อยู่ในช่วงปกติของ **โวลต์** และ **mA·cm⁻²** พอดี
(เทียบกับแถว Voc/Jsc ที่ label ถูกในตารางเดียวกัน: Voc ส่วนใหญ่ 1.0–1.2 V, Jsc ส่วนใหญ่ 15–25 mA cm⁻²)
สรุปว่าเป็น **data-entry error ตอนกรอกตาราง** ในต้นฉบับ (unit label หลุด ไม่ใช่ตัวเลขผิด)

### ปัญหาที่ 2 — mobility หน่วยเป็น m² แทน cm² (ต้อง ×10⁴)

พบใน `Hole mobility` ของ paper `10.1021_acsami.5c16972` (ทั้ง 3 samples) — unit เขียนว่า `m2 V-1 s-1`
ขณะที่ paper อื่นในชุดข้อมูลใช้ `cm2 V-1 s-1` / `cm2/Vs` เป็นมาตรฐาน — 1 m² = 10⁴ cm² จึงต้องคูณค่าด้วย 10⁴


In [21]:
def metric_of(name):
    n = str(name).lower()
    for k, b in [("voc", "Voc"), ("jsc", "Jsc"), ("mobility", "mobility")]:
        if k in n: return b
    return "other"

prop = tables["properties"]
metric = prop["property"].map(metric_of)
u = prop["unit"].astype(str).str.strip()

mislabel_idx = prop.index[(u == "%") & metric.isin(["Voc", "Jsc"])]
mobility_m2_idx = prop.index[(metric == "mobility") & u.str.startswith("m2")]

print(f"Voc/Jsc ติด unit '%'          : {len(mislabel_idx)} แถว")
print(f"mobility หน่วย m2 จริง (x1e4) : {len(mobility_m2_idx)} แถว "
      f"(audit เดิมนับผิดเป็น 7 — 4 ใน นั้นคือ 'cm2 V-1 s-1' ที่ถูกอยู่แล้ว ไม่ใช่ m2)")
prop.loc[list(mislabel_idx) + list(mobility_m2_idx), ["device_id", "property", "value", "unit"]]

Voc/Jsc ติด unit '%'          : 12 แถว
mobility หน่วย m2 จริง (x1e4) : 3 แถว (audit เดิมนับผิดเป็น 7 — 4 ใน นั้นคือ 'cm2 V-1 s-1' ที่ถูกอยู่แล้ว ไม่ใช่ m2)


,device_id,property,value,unit
48,10.1016j.jechem.2021.11.029_Sample1,Voc (RS),1.064,%
49,10.1016j.jechem.2021.11.029_Sample1,Jsc (RS),21.32,%
52,10.1016j.jechem.2021.11.029_Sample1,Voc (FS),1.061,%
53,10.1016j.jechem.2021.11.029_Sample1,Jsc (FS),21.12,%
56,10.1016j.jechem.2021.11.029_Sample1,Average Voc,1.065,%
57,10.1016j.jechem.2021.11.029_Sample1,Average Jsc,20.78,%
68,10.1016j.jechem.2021.11.029_Sample2,Voc (RS),1.118,%
69,10.1016j.jechem.2021.11.029_Sample2,Jsc (RS),21.98,%
72,10.1016j.jechem.2021.11.029_Sample2,Voc (FS),1.122,%
73,10.1016j.jechem.2021.11.029_Sample2,Jsc (FS),22.12,%


## I. `properties` unit ผิด — fix

**⚠️ บั๊กที่เจอระหว่างเช็คก่อน apply:** audit เดิม (§5.3 / "ยัง explore") ใช้ `unit.str.contains("m2 V")`
เพื่อหาแถว mobility หน่วย m² — แต่ regex นี้ไม่ anchor ต้นสตริง เลยไปแมตช์ **`"cm2 V-1 s-1"`** ติดมาด้วย
(เพราะ `"cm2 V-1 s-1"` มี substring `"m2 V"` อยู่ข้างใน ตั้งแต่ตัวที่ 2) ผลคือ audit เดิมนับผิดเป็น **7 แถว**
ทั้งที่จริงมีแค่ **3 แถว** ที่หน่วยเป็น m² จริง ๆ (`10.1021_acsami.5c16972` samples 1-3) — อีก **4 แถว**
(`10_1002_smll_202504237` ×2, `10_1002_solr_202200422` ×2) หน่วยเป็น `cm2 V-1 s-1` **ถูกอยู่แล้ว**

ถ้า apply ตาม audit เดิมตรง ๆ (คูณ 10⁴ ทั้ง 7 แถว) จะทำให้ 4 แถวที่ถูกอยู่แล้วพังไป **หมื่นเท่า**
(เช่น `10_1002_smll_202504237_Sample2` จาก 43.2 กลายเป็น 432000 cm²/(V·s) ซึ่งไม่สมเหตุผลเลยสำหรับ hole mobility)
โค้ดด้านล่างเลยเปลี่ยนเงื่อนไขเป็น `unit.str.startswith("m2")` แทน — แมตช์แค่ 3 แถวที่ผิดจริง

**ตรวจทาน (sanity check):** หลังคูณ 10⁴ ค่า mobility ที่แก้แล้ว (3.67, 4.65, 6.94 cm²/(V·s)) อยู่ในช่วงเดียวกับ
mobility ที่ label ถูกอยู่แล้วของ paper อื่นในชุดข้อมูล (`10_1002_smll_202504237`: 3.46, 43.2 cm²/(V·s))
— ไม่ใช่ค่าที่หลุดโลกผิดปกติ ยืนยันว่าการแก้สมเหตุผล

**สิ่งที่โค้ดทำ:** เพิ่ม 3 คอลัมน์ใหม่ใน `properties` — `value_corrected`, `unit_corrected`, `unit_fix_note`
(ไม่แตะ `value`/`unit` เดิมเลยตามหลักการเดียวกับ E/F/G/H ด้านบน) — กรณี Voc/Jsc แก้แค่ `unit_corrected`
(`value_corrected` เท่ากับ `value` เดิม), กรณี mobility แก้ทั้ง `value_corrected` (×10⁴) และ `unit_corrected`


In [22]:
VOC_JSC_UNIT = {"Voc": "V", "Jsc": "mA cm-2"}   # ตรงกับ unit ที่ถูกอยู่แล้วในตารางนี้

prop["unit_corrected"] = prop["unit"]
prop["value_corrected"] = pd.to_numeric(prop["value"], errors="coerce")
prop["unit_fix_note"] = ""

for idx in mislabel_idx:
    m = metric.at[idx]
    correct_unit = VOC_JSC_UNIT[m]
    prop.at[idx, "unit_corrected"] = correct_unit
    prop.at[idx, "unit_fix_note"] = f"unit เดิม '%' ผิดชนิด แก้เป็น '{correct_unit}' (ไม่แตะค่า value)"

for idx in mobility_m2_idx:
    old_val = prop.at[idx, "value_corrected"]
    new_val = old_val * 1e4
    prop.at[idx, "value_corrected"] = new_val
    prop.at[idx, "unit_corrected"] = prop.at[idx, "unit"].replace("m2", "cm2", 1)
    prop.at[idx, "unit_fix_note"] = f"หน่วย m2 -> cm2 (x1e4): {old_val:g} -> {new_val:g}"

n_fixed = len(mislabel_idx) + len(mobility_m2_idx)
print(f"แก้แล้ว {n_fixed} แถว ({len(mislabel_idx)} unit mislabel + {len(mobility_m2_idx)} mobility m2->cm2)")
prop.loc[list(mislabel_idx) + list(mobility_m2_idx),
         ["device_id", "property", "value", "unit", "value_corrected", "unit_corrected", "unit_fix_note"]]

แก้แล้ว 15 แถว (12 unit mislabel + 3 mobility m2->cm2)


,device_id,property,value,unit,value_corrected,unit_corrected,unit_fix_note
48,10.1016j.jechem.2021.11.029_Sample1,Voc (RS),1.064,%,1.064,V,unit เดิม '%' ผิดชนิด แก้เป็น 'V' (ไม่แตะค่า value)
49,10.1016j.jechem.2021.11.029_Sample1,Jsc (RS),21.32,%,21.320,mA cm-2,unit เดิม '%' ผิดชนิด แก้เป็น 'mA cm-2' (ไม่แตะค่า value)
52,10.1016j.jechem.2021.11.029_Sample1,Voc (FS),1.061,%,1.061,V,unit เดิม '%' ผิดชนิด แก้เป็น 'V' (ไม่แตะค่า value)
53,10.1016j.jechem.2021.11.029_Sample1,Jsc (FS),21.12,%,21.120,mA cm-2,unit เดิม '%' ผิดชนิด แก้เป็น 'mA cm-2' (ไม่แตะค่า value)
56,10.1016j.jechem.2021.11.029_Sample1,Average Voc,1.065,%,1.065,V,unit เดิม '%' ผิดชนิด แก้เป็น 'V' (ไม่แตะค่า value)
57,10.1016j.jechem.2021.11.029_Sample1,Average Jsc,20.78,%,20.780,mA cm-2,unit เดิม '%' ผิดชนิด แก้เป็น 'mA cm-2' (ไม่แตะค่า value)
68,10.1016j.jechem.2021.11.029_Sample2,Voc (RS),1.118,%,1.118,V,unit เดิม '%' ผิดชนิด แก้เป็น 'V' (ไม่แตะค่า value)
69,10.1016j.jechem.2021.11.029_Sample2,Jsc (RS),21.98,%,21.980,mA cm-2,unit เดิม '%' ผิดชนิด แก้เป็น 'mA cm-2' (ไม่แตะค่า value)
72,10.1016j.jechem.2021.11.029_Sample2,Voc (FS),1.122,%,1.122,V,unit เดิม '%' ผิดชนิด แก้เป็น 'V' (ไม่แตะค่า value)
73,10.1016j.jechem.2021.11.029_Sample2,Jsc (FS),22.12,%,22.120,mA cm-2,unit เดิม '%' ผิดชนิด แก้เป็น 'mA cm-2' (ไม่แตะค่า value)


## สรุป + save audit + verify (จบช่วง 2 — ยังไม่เขียนดิสก์ ต่อด้วยช่วง 3 ก่อน)

In [23]:
audit = pd.DataFrame(audit_rows,
                     columns=["table", "column", "device_id", "raw",
                              "parsed", "vmin", "vmax", "extra", "note"])

ba = (audit.groupby(["table", "column", "raw", "parsed", "vmin", "vmax", "extra"],
                    dropna=False)
           .size().rename("n_rows").reset_index()
           .rename(columns={"raw": "before", "parsed": "after",
                            "extra": "kind/flag"})
           .sort_values(["table", "column", "before"])
           .reset_index(drop=True))
print(f"คู่ before -> after ไม่ซ้ำ: {len(ba)} | ครอบคลุม {ba.n_rows.sum()} แถวที่ถูกตีความ")
ba

คู่ before -> after ไม่ซ้ำ: 23 | ครอบคลุม 79 แถวที่ถูกตีความ


,table,column,before,after,vmin,vmax,kind/flag,n_rows
0,layers,thickness_nm,1200 Å,120.00,120.0,120.0,Cathode,2
1,layers,thickness_nm,"4, 100",104.00,4.0,100.0,Cathode,2
2,layers,thickness_nm,400 Å,40.00,40.0,40.0,ETL treatment (1),2
3,layers,thickness_nm,400-450,425.00,400.0,450.0,PSK,4
4,layers,thickness_nm,90 Å,9.00,9.0,9.0,ETL treatment (2),2
5,stability,humidity_pct,25 ± 5,25.00,20.0,30.0,,2
6,stability,humidity_pct,30-50,40.00,30.0,50.0,,3
7,stability,humidity_pct,45–55,50.00,45.0,55.0,,1
8,stability,humidity_pct,50 - 60,55.00,50.0,60.0,,1
9,stability,humidity_pct,50 ± 5,50.00,45.0,55.0,,2


In [24]:
audit.to_csv(AUDIT / "value_parse_audit.csv", index=False)
print("บันทึก audit แล้ว:", AUDIT / "value_parse_audit.csv", f"({len(audit)} แถว)")
audit.groupby(["table", "column"]).size().rename("n_rows").to_frame()

บันทึก audit แล้ว: D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\audit\value_parse_audit.csv (79 แถว)


n_rows
table     column               
layers    thickness_nm       12
stability humidity_pct       12
          retention_pct       1
          temperature_c       6
vendors   purity             48

In [25]:
raw_ok = all(tables[n][c].fillna("").equals(tables_after_text[n][c].fillna(""))
             for n in tables_after_text for c in tables_after_text[n].columns)
rows_ok = all(len(tables[n]) == len(tables_after_text[n]) for n in tables)

n_bad_total = 0
unhandled = []
for name, cols in NUM_COLS.items():
    for col in cols:
        s = tables[name][col].fillna("").str.strip()
        bad_idx = s[(s != "") & ~s.map(is_num)].index
        n_bad_total += len(bad_idx)
        for i in bad_idx:
            note_cols = [c for c in tables[name].columns
                         if c.startswith(col) and c != col]
            vals = [str(tables[name].at[i, c]) for c in note_cols]
            handled = any(v not in ("", "nan", "None") for v in vals)
            flagged_bad = any("parse ไม่ได้" in v or v == "unrecognized" for v in vals)
            if not handled or flagged_bad:
                unhandled.append((name, col, int(i), s[i]))

print("1) คอลัมน์เดิมไม่ถูกแก้ระหว่างช่วง 2 :", raw_ok)
print("2) จำนวนแถวเท่าเดิมทุกตาราง            :", rows_ok)
print(f"3) เคส non-numeric {n_bad_total} แถว | ใน audit {len(audit)} แถว | หลุด: {len(unhandled)}")
assert raw_ok and rows_ok and n_bad_total == len(audit) and not unhandled, "VERIFY FAILED"
print("ช่วง 2 (value-format) verify ผ่านหมด ✓")

1) คอลัมน์เดิมไม่ถูกแก้ระหว่างช่วง 2 : True
2) จำนวนแถวเท่าเดิมทุกตาราง            : True
3) เคส non-numeric 79 แถว | ใน audit 79 แถว | หลุด: 0
ช่วง 2 (value-format) verify ผ่านหมด ✓


# ช่วง 3 — Prepare properties (แตกชื่อ + แปลงสเกล FF) → `processed/prepared/`

ต่อจากช่วง 1+2 — ใช้ `tables` ใน memory ต่อเลย (ไม่มีโฟลเดอร์กลางระหว่างช่วง 1→2→3) "
เขียนดิสก์จุดเดียวที่ท้ายช่วง 3 นี้ ส่วน `eda_prepare.ipynb` เหลือแค่ EDA (อ่านต่อจาก `processed/prepared/`)

## 3.0 ทำไมต้องแตกชื่อ property

ตอนนี้ `properties.property` มี 91 ชื่อ ซึ่งจริง ๆ เป็นการเอา **5 มิติ** มาต่อกันเป็น string เดียว:

```
"Champion PCE (FS)"  =  base: PCE       + stat: champion + scan: FS
"HTL grain size"     =  base: Grain size + layer: HTL
"Hole VTFL"          =  base: VTFL      + carrier: hole
```

ถ้าไม่แตก มิติพวกนี้จะฝังอยู่ในชื่อ ทำให้ pivot/slice ไม่ได้ (เช่น "เอาเฉพาะ champion + reverse scan")
และ 91 ชื่อจะกลายเป็น 91 features กระจัดกระจายตอนทำ ML — แตกแล้วเหลือ **base 46 ตัว** + คอลัมน์กรอง 4 ตัว

| มิติ | รูปแบบในชื่อ | ค่า | หมายเหตุ |
|---|---|---|---|
| `stat` | คำนำหน้า Champion/Average/Avg/Certified/Verified | เก็บ**ตามที่เขียน** (lower-case) | ⚠️ `avg` ≠ `average`, `verified` ≠ `certified` — **ตั้งใจไม่รวม** เพราะเป็นกลุ่มรอยืนยัน มี `STAT_MERGE_PENDING` เตรียมไว้ เปิดใช้เมื่อยืนยันแล้ว |
| `scan` | วงเล็บท้าย (RS)/(FS) | RS / FS | reverse / forward scan ของ J-V |
| `layer` | คำนำหน้า HTL/PSK หรือวงเล็บท้าย (HTL)/(PVK) | HTL / PSK | `PVK→PSK` เป็นแค่ตัวสะกด (ไม่ใช่ semantic) |
| `carrier` | คำนำหน้า Electron/Hole | electron / hole | ใช้กับ VTFL, trap density, mobility |

หลังตัด prefix/suffix แล้ว base ที่เหลือบางตัวสะกดต่างกันแค่ตัวพิมพ์ → ใช้กติกา **variant-merge แบบเดียวกับช่วง 1** (majority vote, case/วรรคตอนเท่านั้น) ปิดท้าย

## 3.0b กติกาแปลงสเกล FF

FF ถูกรายงาน 2 สเกล — สัดส่วน 0.71–0.78 (จาก 2 DOI) กับเปอร์เซ็นต์ 55–84.7
- กติกา: แถวที่ `base_property == "FF"` และค่า ≤ 1 → **× 100** (บันทึกใน `value_note` + audit)
- ช่องว่างระหว่างสองกลุ่ม (0.78 กับ 55) กว้างมาก — ไม่มีเคสก้ำกึ่ง
- ทำเฉพาะ FF เท่านั้น: `Voc` ค่า ~1.0–1.2 เป็นปกติ (หน่วย V) และ `HI` ค่าน้อย ๆ อาจ ≤ 1 ได้จริง — ห้ามเหมารวม
- ค่าแปลงแล้วอยู่ในคอลัมน์ใหม่ `value_parsed` (float) — **`value` เดิมไม่ถูกแตะ**


In [26]:
DST3 = ROOT / "processed" / "prepared"      # โฟลเดอร์ output เดียวของทั้ง pipeline

pd.options.display.max_rows = 120
tables_after_values = {name: df.copy() for name, df in tables.items()}  # snapshot ไว้เทียบใน §3.4
prep_audit = []
{name: df.shape for name, df in tables.items()}

{'devices': (44, 4),
 'layers': (352, 10),
 'materials': (543, 8),
 'process': (702, 9),
 'properties': (563, 9),
 'stability': (62, 19),
 'vendors': (644, 8)}

## 3.1 แตกชื่อ property — explore

In [27]:
n_unique = tables["properties"]["property"].nunique()
print(f"properties.property มี {n_unique} ชื่อไม่ซ้ำ — แต่ละชื่อผสมหลายมิติ (base+stat+scan+layer+carrier) เข้าด้วยกัน")
print("ตัวอย่างที่เห็นชัด:")
for n in ["Champion PCE (FS)", "HTL grain size", "Hole VTFL", "Average Voc", "Certified PCE (RS)"]:
    hits = tables["properties"][tables["properties"]["property"] == n]
    if len(hits):
        print(f"  {n!r}: {len(hits)} แถว")
print()
tables["properties"]["property"].value_counts().head(15)

properties.property มี 91 ชื่อไม่ซ้ำ — แต่ละชื่อผสมหลายมิติ (base+stat+scan+layer+carrier) เข้าด้วยกัน
ตัวอย่างที่เห็นชัด:
  'Champion PCE (FS)': 4 แถว
  'HTL grain size': 3 แถว
  'Hole VTFL': 2 แถว
  'Average Voc': 13 แถว
  'Certified PCE (RS)': 1 แถว



property
VTFL             26
Champion PCE     24
Rs               22
Grain size       20
Champion Voc     20
Champion Jsc     20
Champion FF      20
Trap density     18
Contact angle    17
VBM              15
Rrec             15
Average PCE      14
Average Voc      13
Average Jsc      13
Average FF       13
Name: count, dtype: int64

## 3.1 แตกชื่อ property — fix

In [28]:
def variant_key(s: str) -> str:
    return re.sub(r"[\W_]", "", s.lower())

def decompose(name: str):
    """ตัด stat -> scan -> layer -> carrier ออกจากชื่อ เหลือ base"""
    base, stat, scan, layer, carrier = name, "", "", "", ""
    m = re.match(r"^(Champion|Average|Avg|Certified|Verified)\s+(.+)$", base)
    if m:
        stat, base = m[1].lower(), m[2]
    m = re.match(r"^(.+?)\s*\((RS|FS)\)$", base)
    if m:
        base, scan = m[1], m[2]
    m = re.match(r"^(HTL|PSK|PVK)\s+(.+)$", base)
    if m:
        layer, base = m[1], m[2]
    else:
        m = re.match(r"^(.+?)\s*\((HTL|PVK|PSK)\)$", base)
        if m:
            base, layer = m[1], m[2]
    layer = {"PVK": "PSK"}.get(layer, layer)
    m = re.match(r"^(Electron|Hole)\s+(.+)$", base)
    if m:
        carrier, base = m[1].lower(), m[2]
    return base, stat, scan, layer, carrier

STAT_MERGE_PENDING = {"avg": "average", "verified": "certified"}

prop = tables["properties"]
parts = prop["property"].map(decompose)
for i, col in enumerate(["base_property", "stat", "scan", "layer", "carrier"]):
    prop[col] = parts.map(lambda t, i=i: t[i])

PREFERRED_BASE = {"grainsize": "Grain size"}
groups = {}
for val, n in Counter(prop["base_property"]).items():
    groups.setdefault(variant_key(val), []).append((val, n))
base_map = {}
for k, variants in groups.items():
    canonical = PREFERRED_BASE.get(k) or max(variants, key=lambda v: (v[1], v[0]))[0]
    base_map.update({val: canonical for val, n in variants if val != canonical})
prop["base_property"] = prop["base_property"].replace(base_map)

print(f"ชื่อ property {prop['property'].nunique()} แบบ -> base_property {prop['base_property'].nunique()} แบบ")
print("case-merge หลังแตก:", base_map)

decomp_map = (prop.groupby(["property", "base_property", "stat", "scan", "layer", "carrier"])
                  .size().rename("n_rows").reset_index()
                  .sort_values(["base_property", "stat", "scan", "layer", "carrier"])
                  .reset_index(drop=True))
decomp_map

ชื่อ property 91 แบบ -> base_property 46 แบบ
case-merge หลังแตก: {'Contact Angle': 'Contact angle', 'grain size': 'Grain size', 'Trap Density': 'Trap density', 'AFM rms': 'AFM RMS'}


,property,base_property,stat,scan,layer,carrier,n_rows
0,AFM RMS,AFM RMS,,,,,4
1,AFM rms (HTL),AFM RMS,,,HTL,,1
2,AFM rms (PVK),AFM RMS,,,PSK,,1
3,Bandgap,Bandgap,,,,,1
4,Carrier concentration,Carrier concentration,,,,,2
5,Conductivity,Conductivity,,,,,4
6,Contact angle,Contact angle,,,,,17
7,Contact Angle (HTL),Contact angle,,,HTL,,1
8,Crec,Crec,,,,,3
9,Defect density,Defect density,,,,,2


## 3.2 `value_parsed` + แปลงสเกล FF — explore

In [29]:
prop = tables["properties"]
ff_check = prop[prop["base_property"] == "FF"].copy()
ff_check["v"] = pd.to_numeric(ff_check["value"], errors="coerce")
print(f"FF ทั้งหมด {len(ff_check)} แถว")
print(f"  เป็นสัดส่วน (<=1, ต้องคูณ 100)  : {int((ff_check['v']<=1).sum())} แถว")
print(f"  เป็น % อยู่แล้ว (>1)            : {int((ff_check['v']>1).sum())} แถว")
ff_check.loc[ff_check["v"] <= 1, ["device_id", "property", "value"]]

FF ทั้งหมด 67 แถว
  เป็นสัดส่วน (<=1, ต้องคูณ 100)  : 8 แถว
  เป็น % อยู่แล้ว (>1)            : 59 แถว


,device_id,property,value
2,10.1002admi.202500231_Sample1,FF,0.76
14,10.1002admi.202500231_Sample2,FF,0.75
26,10.1002admi.202500231_Sample3,FF,0.78
38,10.1002admi.202500231_Sample4,FF,0.79
349,10_1016_j_solener_2025_113286_Sample1,Average FF,0.69
353,10_1016_j_solener_2025_113286_Sample1,Champion FF,0.71
364,10_1016_j_solener_2025_113286_Sample2,Average FF,0.72
368,10_1016_j_solener_2025_113286_Sample2,Champion FF,0.72


## 3.2 `value_parsed` + แปลงสเกล FF — fix

In [30]:
prop["value_parsed"] = pd.to_numeric(prop["value"], errors="coerce")
prop["value_note"] = ""

ff_mask = (prop["base_property"] == "FF") & prop["value_parsed"].notna() & (prop["value_parsed"] <= 1)
ff_ba = prop.loc[ff_mask, ["device_id", "property", "value_parsed"]].rename(
    columns={"value_parsed": "before"})
prop.loc[ff_mask, "value_parsed"] = prop.loc[ff_mask, "value_parsed"] * 100
prop.loc[ff_mask, "value_note"] = "FF scale: fraction -> % (x100)"
ff_ba["after"] = prop.loc[ff_mask, "value_parsed"].values

for _, r in ff_ba.iterrows():
    prep_audit.append({"table": "properties", "column": "value_parsed",
                       "device_id": r["device_id"], "property": r["property"],
                       "before": r["before"], "after": r["after"],
                       "note": "FF scale x100"})

n_ff = int((prop["base_property"] == "FF").sum())
print(f"FF ทั้งหมด {n_ff} แถว | แปลงสเกล {int(ff_mask.sum())} แถว")
print("DOI ที่ได้รับผล:",
      sorted(set(ff_ba["device_id"].str.replace(r"_Sample\d+$", "", regex=True))))
ff_ba.reset_index(drop=True)

FF ทั้งหมด 67 แถว | แปลงสเกล 8 แถว
DOI ที่ได้รับผล: ['10.1002admi.202500231', '10_1016_j_solener_2025_113286']


,device_id,property,before,after
0,10.1002admi.202500231_Sample1,FF,0.76,76.0
1,10.1002admi.202500231_Sample2,FF,0.75,75.0
2,10.1002admi.202500231_Sample3,FF,0.78,78.0
3,10.1002admi.202500231_Sample4,FF,0.79,79.0
4,10_1016_j_solener_2025_113286_Sample1,Average FF,0.69,69.0
5,10_1016_j_solener_2025_113286_Sample1,Champion FF,0.71,71.0
6,10_1016_j_solener_2025_113286_Sample2,Average FF,0.72,72.0
7,10_1016_j_solener_2025_113286_Sample2,Champion FF,0.72,72.0


## 3.3 เขียนดิสก์ครั้งเดียว → `processed/prepared/` + audit

**จุดเดียวที่ pipeline เขียนตารางลงดิสก์** — `prepared/` คือผลรวมของทั้ง 3 ช่วง: ข้อความ canonical (ช่วง 1) + คอลัมน์ `_parsed`/`_note` (ช่วง 2) + property 5 มิติ และ FF สเกลเดียว (ช่วง 3)

- `properties` ได้คอลัมน์ใหม่ 7 คอลัมน์ (base/stat/scan/layer/carrier/value_parsed/value_note) + 3 คอลัมน์จาก I (value_corrected/unit_corrected/unit_fix_note)
- `processed/audit/property_decomposition.csv` — mapping ชื่อ → 5 มิติ ครบ 91 ชื่อ
- `processed/audit/prepare_audit.csv` — ทุกแถวที่ค่าถูกแปลง (FF ×100)


In [31]:
DST3.mkdir(parents=True, exist_ok=True)
for name, df in tables.items():
    df.to_csv(DST3 / f"{name}.csv", index=False)
decomp_map.to_csv(AUDIT / "property_decomposition.csv", index=False)
pd.DataFrame(prep_audit).to_csv(AUDIT / "prepare_audit.csv", index=False)

print("บันทึกแล้ว:", DST3)
print("  property_decomposition.csv:", len(decomp_map), "ชื่อ")
print("  prepare_audit.csv:", len(prep_audit), "แถว (FF conversions)")

บันทึกแล้ว: D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\prepared
  property_decomposition.csv: 91 ชื่อ
  prepare_audit.csv: 8 แถว (FF conversions)


## 3.4 Verify จากไฟล์ที่เซฟจริงบนดิสก์ — 4 เงื่อนไข (จุดตรวจรวมศูนย์ของทั้ง pipeline)

1. **ไฟล์บนดิสก์ตรงกับตารางใน memory** ทุกคอลัมน์ทุกเซลล์ — การเขียนครั้งเดียวไม่เพี้ยน
2. **จำนวนแถวเท่ากับข้อมูลดิบ `processed/normalize/`** ทุกตาราง — ไม่มีแถวหาย/เกินตลอดช่วง 1→3
3. ทุกแถว property ถูกแตก (`base_property` ไม่ว่าง)
4. FF ใน `value_parsed` เป็น % ล้วน (30–100)


In [32]:
svn = {p.stem: pd.read_csv(p) for p in sorted(DST3.glob("*.csv"))}

mismatches = []
for n in tables:
    for c in tables[n].columns:
        mem, dsk = tables[n][c], svn[n][c]
        if pd.api.types.is_numeric_dtype(dsk):
            m = pd.to_numeric(mem, errors="coerce")
            same = bool((m.isna() == dsk.isna()).all()) and \
                   bool(((m - dsk).abs().fillna(0) < 1e-9).all())
        else:
            same = mem.fillna("").astype(str).equals(dsk.fillna("").astype(str))
        if not same:
            mismatches.append(f"{n}.{c}")

disk_ok = not mismatches
rows_ok = all(len(svn[n]) == len(tables_raw[n]) for n in svn)
base_ok = bool((svn["properties"]["base_property"].fillna("") != "").all())
ffv = svn["properties"].loc[svn["properties"]["base_property"] == "FF", "value_parsed"].dropna()
ff_ok = bool(((ffv > 30) & (ffv <= 100)).all())

print("1) ไฟล์บนดิสก์ตรงกับตารางใน memory       :", disk_ok, mismatches if mismatches else "")
print("2) จำนวนแถวเท่ากับข้อมูลดิบ processed/normalize :", rows_ok)
print("3) base_property ครบทุกแถว                 :", base_ok)
print(f"4) FF เป็น % ล้วน ({ffv.min():.1f}-{ffv.max():.1f})           :", ff_ok)
assert disk_ok and rows_ok and base_ok and ff_ok, "VERIFY FAILED"
print("ผ่านครบ 4 เงื่อนไข ✓ — processed/prepared/ พร้อมใช้")

1) ไฟล์บนดิสก์ตรงกับตารางใน memory       : True 
2) จำนวนแถวเท่ากับข้อมูลดิบ processed/normalize : True
3) base_property ครบทุกแถว                 : True
4) FF เป็น % ล้วน (65.4-84.7)           : True
ผ่านครบ 4 เงื่อนไข ✓ — processed/prepared/ พร้อมใช้


# ยัง explore อย่างเดียว — ยังไม่มี fix จริง

หัวข้อด้านล่างพอร์ตมาจาก `column_audit.ipynb` เหมือนกัน แต่ยังไม่มีจุดไหนใน pipeline apply เป็นคอลัมน์จริง
(§ Voc/Jsc/mobility unit ที่เคยอยู่ตรงนี้ ย้ายขึ้นไปเป็นหัวข้อ **I** ในช่วง 2 แล้ว แก้จริงแล้ว)

## `process.process_type` canonical mapping (snake_case) — เสนอไว้ ยังไม่ apply

In [33]:
PROCESS_CANON = {
 "spincoating":"spin_coating", "spin coating":"spin_coating",
 "annealing":"annealing", "postannealing":"post_annealing",
 "thermalevaporation":"thermal_evaporation", "thermal evaporation":"thermal_evaporation",
 "electronbeamevaporation":"ebeam_evaporation",
 "antisolvent":"antisolvent",
 "wash":"wash", "ethanolwash":"wash", "dry":"dry",
 "plasmatreat":"plasma_treatment", "plasma treat":"plasma_treatment",
 "uv/ozone treat":"uv_ozone_treatment", "uv treat":"uv_treatment", "treatment":"treatment",
 "stir":"stir", "magnetic stir":"stir",
 "mix":"mixing", "mixing":"mixing",
 "sonication":"sonication", "dip-coating":"dip_coating", "dissolve":"dissolve",
 "ph adjustment":"ph_adjustment",
 "sputtering":"sputtering", "rfsputtering":"sputtering", "magnetronsputtering":"sputtering",
 "presputtering":"pre_sputtering", "pre-sputtering":"pre_sputtering",
 "ald":"ald",
}
def map_report(series, canon):
    vc = series.dropna().astype(str).str.strip().value_counts()
    df = vc.rename_axis("raw").reset_index(name="count")
    df["canonical"] = df["raw"].str.lower().map(canon)
    return df

pt = map_report(tables["process"]["process_type"], PROCESS_CANON)
unmapped = pt[pt["canonical"].isna()]
print(f"raw {pt['raw'].nunique()} แบบ -> canonical {pt['canonical'].nunique()} แบบ | "
      f"map ไม่เจอ {len(unmapped)} แบบ ({unmapped['count'].sum()} rows)")
pt

raw 26 แบบ -> canonical 21 แบบ | map ไม่เจอ 0 แบบ (0 rows)


,raw,count,canonical
0,Spin coating,213,spin_coating
1,Annealing,161,annealing
2,Thermal evaporation,82,thermal_evaporation
3,Wash,52,wash
4,AntiSolvent,42,antisolvent
5,Dry,32,dry
6,Plasma Treat,21,plasma_treatment
7,UV/Ozone Treat,14,uv_ozone_treatment
8,Stir,8,stir
9,Sonication,4,sonication


## `stability.environment` → atmosphere grouping — เสนอไว้ ยังไม่ apply

In [34]:
def atmosphere(x):
    s = str(x).lower()
    if "n2" in s or "nitrogen" in s: return "N2/inert"
    if "ambient" in s or "air" in s or "humidity" in s: return "ambient/air"
    if "thermal" in s: return "thermal"
    if "mpp" in s or "operational" in s: return "operational(MPP)"
    if "isos" in s: return "ISOS (unspecified atm.)"
    return "other"
env = tables["stability"]["environment"].dropna().str.strip()
edf = env.value_counts().rename_axis("raw").reset_index(name="count")
edf["atmosphere"] = edf["raw"].map(atmosphere)
edf.sort_values(["atmosphere","count"], ascending=[True,False])

,raw,count,atmosphere
3,ISOS-L2,4,ISOS (unspecified atm.)
18,ISOS-L-3,1,ISOS (unspecified atm.)
0,N2,11,N2/inert
4,N2/ISOS-L-2I,4,N2/inert
5,Nitrogen (N2),3,N2/inert
8,N2/ISOS-D-2,3,N2/inert
12,N2 atmosphere,2,N2/inert
15,"N2 glovebox, AM1.5G 1-sun illumination",2,N2/inert
17,N2 storage,1,N2/inert
21,N2/ISOS-L-2I (with top passivation),1,N2/inert


## Unit audit — เช็คหน่วยทุกคอลัมน์ (ยังเป็นแค่ audit)

1. หน่วยฝังในชื่อ header — ค่าอยู่ในช่วงสมเหตุผลไหม
2. ปริมาณเดียวกันใน `properties` — หน่วยตรงกันทุกแถวไหม
3. micro-sign vs greek-mu

(ค่าที่ติดหน่วยผิด value-level — Voc/Jsc `%`, mobility m² — ย้ายไปแก้จริงแล้วที่หัวข้อ **I** ในช่วง 2)

In [35]:
hdr_units = {
 ("layers","thickness_nm","nm"):(0,2000),
 ("stability","temperature_c","C"):(-20,700),
 ("stability","humidity_pct","%"):(0,100),
 ("stability","time_hr","hr"):(0,50000),
 ("stability","retention_pct","%"):(0,110),
 ("stability","initial_pce","%"):(0,30),
 ("stability","final_pce","%"):(0,30),
}
rowsx = []
for (tab,col,unit),(lo,hi) in hdr_units.items():
    s = pd.to_numeric(tables[tab][col], errors="coerce").dropna()
    oob = s[(s<lo)|(s>hi)]
    rowsx.append({"table":tab,"column":col,"unit":unit,"n":int(len(s)),
                 "min":round(float(s.min()),3),"max":round(float(s.max()),3),
                 "expected_range":f"[{lo},{hi}]","n_out_of_range":int(len(oob)),
                 "status":"OK" if len(oob)==0 else "CHECK"})
unit_header_check = pd.DataFrame(rowsx)
unit_header_check

,table,column,unit,n,min,max,expected_range,n_out_of_range,status
0,layers,thickness_nm,nm,93,5.00,715.00,"[0,2000]",0,OK
1,stability,temperature_c,C,42,25.00,85.00,"[-20,700]",0,OK
2,stability,humidity_pct,%,14,0.00,60.00,"[0,100]",0,OK
3,stability,time_hr,hr,59,0.83,5000.00,"[0,50000]",0,OK
4,stability,retention_pct,%,49,0.40,100.85,"[0,110]",0,OK
5,stability,initial_pce,%,4,16.25,17.11,"[0,30]",0,OK
6,stability,final_pce,%,4,10.50,14.50,"[0,30]",0,OK


In [36]:
def metric_of_full(name):
    n = str(name).lower()
    for k,b in [("pce","PCE"),("voc","Voc"),("jsc","Jsc"),("ff","FF"),
                ("band","bandgap"),("vbm","VBM"),("fermi","Fermi level"),
                ("work function","Work function"),("mobility","mobility"),
                ("trap","trap density"),("grain","grain size"),
                ("contact angle","contact angle"),("rms","roughness RMS"),
                ("conductivity","conductivity"),("lifetime","carrier lifetime"),
                ("eqe","EQE"),("spo","SPO")]:
        if k in n: return b
    if re.search(r"\brs\b", n) or "series resist" in n: return "series R (Rs)"
    if "rsh" in n or "shunt" in n: return "shunt R (Rsh)"
    return "other"

pp = tables["properties"].copy()
pp["metric"] = pp["property"].map(metric_of_full)
pp["u"] = pp["unit"].astype(str).str.strip()
rowsy = []
for m, grp in pp.dropna(subset=["unit"]).groupby("metric"):
    uv = grp["u"].value_counts()
    rowsy.append({"metric":m,"n":int(uv.sum()),"n_distinct_units":int(uv.size),
                 "units":", ".join(f"{u} x{c}" for u,c in uv.items()),
                 "flag":"MULTI-UNIT" if uv.size>1 else "OK"})
prop_unit_consistency = (pd.DataFrame(rowsy)
    .sort_values(["flag","n"], ascending=[True,False]).reset_index(drop=True))
prop_unit_consistency

,metric,n,n_distinct_units,units,flag
0,other,101,12,"V x35, ns x12, % x12, eV x11, Ω x8, nm x6, Ohm x5, cm-3 x4, F x3, Ω cm² x3, cm-3 eV-1 x1, cm-3 e/V x1",MULTI-UNIT
1,Jsc,79,2,"mA cm-2 x73, % x6",MULTI-UNIT
2,Voc,67,2,"V x61, % x6",MULTI-UNIT
3,grain size,27,2,"nm x26, µm x1",MULTI-UNIT
4,series R (Rs),23,6,"Ohm x6, Ω x6, Ohm/cm2 x4, Ω cm² x3, Ω/sq x2, Ω cm2 x2",MULTI-UNIT
5,contact angle,18,2,"deg x14, degrees x4",MULTI-UNIT
6,mobility,9,3,"cm2 V-1 s-1 x4, m2 V-1 s-1 x3, cm2/Vs x2",MULTI-UNIT
7,PCE,73,1,% x73,OK
8,FF,59,1,% x59,OK
9,trap density,26,1,cm-3 x26,OK


## micro-sign `µ` (U+00B5) vs greek-mu `μ` (U+03BC) — เช็คใน raw (ก่อน unicode-fix ช่วง 1)

In [37]:
MICRO, MU = chr(0x00B5), chr(0x03BC)
rowsz = []
for tab in ["materials","process","properties"]:
    for val in pd.read_csv(f"processed/normalize/{tab}.csv")["unit"].dropna().astype(str).unique():
        if MICRO in val or MU in val:
            rowsz.append({"table":f"{tab}.unit","raw_unit":val,
                         "micro_sign_U+00B5": MICRO in val,
                         "greek_mu_U+03BC": MU in val,
                         "codepoints":" ".join(f"U+{ord(c):04X}" for c in val if ord(c)>127)})
micro_check = pd.DataFrame(rowsz)
micro_check

,table,raw_unit,micro_sign_U+00B5,greek_mu_U+03BC,codepoints
0,materials.unit,μL,False,True,U+03BC
1,materials.unit,µL,True,False,U+00B5
2,materials.unit,µl,True,False,U+00B5
3,process.unit,"µL, sec",True,False,U+00B5
4,process.unit,"μL, sec",False,True,U+03BC
5,process.unit,μL,False,True,U+03BC
6,process.unit,"µl, sec",True,False,U+00B5
7,process.unit,µL,True,False,U+00B5
8,properties.unit,μm,False,True,U+03BC


## Export unit reports

In [38]:
os.makedirs("processed/audit", exist_ok=True)
prop_unit_consistency.to_csv("processed/audit/unit_consistency_properties.csv", index=False)
micro_check.to_csv("processed/audit/unit_micro_sign.csv", index=False)
print("wrote -> processed/audit/{unit_consistency_properties, unit_micro_sign}.csv")
print("(unit_value_errors.csv เดิมถูกแทนที่ด้วย fix จริงในหัวข้อ I ของช่วง 2 แล้ว — ไม่ export ซ้ำที่นี่)")

wrote -> processed/audit/{unit_consistency_properties, unit_micro_sign}.csv
(unit_value_errors.csv เดิมถูกแทนที่ด้วย fix จริงในหัวข้อ I ของช่วง 2 แล้ว — ไม่ export ซ้ำที่นี่)


## Cross-table consistency — ชื่อวัสดุข้ามตารางตรงกันไหม

วัสดุตัวเดียวกันควรสะกดเหมือนกันใน `layers`, `materials`, `vendors`. หา normalized-key ที่มีหลายสะกด "ข้ามตาราง"
(เช็คหลัง fix H ด้านบนแล้ว — ถ้ายังเจอ แปลว่า per-column merge ในช่วง 1 ไม่พอ ต้องดูข้ามตารางเพิ่ม)

In [39]:
frames = []
for name in ["layers","materials","vendors"]:
    s = tables[name]["material"].dropna().astype(str).str.strip()
    frames.append(pd.DataFrame({"table": name, "material": s, "key": s.map(tkey)}))
allm = pd.concat(frames, ignore_index=True)
rowsw = []
for k, g in allm.groupby("key"):
    spell = sorted(g["material"].unique())
    if len(spell) > 1:
        rowsw.append({"key": k, "n_spellings": len(spell),
                     "spellings": "  |  ".join(spell),
                     "tables": ", ".join(sorted(g["table"].unique()))})
cross = pd.DataFrame(rowsw, columns=["key","n_spellings","spellings","tables"])
if len(cross):
    cross = cross.sort_values("n_spellings", ascending=False).reset_index(drop=True)
print(f"{len(cross)} วัสดุที่สะกดไม่ตรงกัน (normalize แล้ว key เดียวกันแต่หลายสะกด)")
cross if len(cross) else "✅ ชื่อวัสดุสะกดสม่ำเสมอ (ภายใต้ normalize) ทั้งใน layers/materials/vendors"

0 วัสดุที่สะกดไม่ตรงกัน (normalize แล้ว key เดียวกันแต่หลายสะกด)


'✅ ชื่อวัสดุสะกดสม่ำเสมอ (ภายใต้ normalize) ทั้งใน layers/materials/vendors'